In [1]:
# ================= Cell 1 =================
# 安裝必要的套件 (如果在 Colab 執行，建議先執行這行；Kaggle 通常已內建)
# !pip install transformers datasets scikit-learn pandas torch

import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW 
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import LabelEncoder
from tqdm.auto import tqdm

# 設定隨機種子以確保結果可重現
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [2]:
# ================= Cell 2 =================
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

DATA_DIR = '/kaggle/input/competitions/map-charting-student-math-misunderstandings/' 

train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

# 確保空值變成字串 'NA'
train_df['Misconception'] = train_df['Misconception'].fillna('NA')

# 將 Category 和 Misconception 合併
train_df['target_label'] = train_df['Category'] + ':' + train_df['Misconception']

# 將 Question, Answer, Explanation 拼接成一個完整的上下文
def create_input_text(row):
    return f"Question: {row['QuestionText']} [SEP] Answer: {row['MC_Answer']} [SEP] Explanation: {row['StudentExplanation']}"

train_df['input_text'] = train_df.apply(create_input_text, axis=1)
test_df['input_text'] = test_df.apply(create_input_text, axis=1)

# Label Encoding (讓模型學習合併後的完整字串)
label_encoder = LabelEncoder()
train_df['label'] = label_encoder.fit_transform(train_df['target_label'])
num_labels = len(label_encoder.classes_)

print(f"Total number of full classes (Category:Misconception): {num_labels}")

# 🎯 修正：拿掉 stratify=train_df['label']，改為純隨機切分，避免 1 筆資料的類別報錯
train_data, val_data = train_test_split(train_df, test_size=0.15, random_state=42)

Total number of full classes (Category:Misconception): 65


In [3]:
# ================= Cell 3 =================
# 使用你成功掛載的 MathBERT 離線路徑！
MODEL_NAME = '/kaggle/input/datasets/huangtzuchen/my-mathbert-weights/mathbert-local'
MAX_LEN = 256
BATCH_SIZE = 16

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class MathMisconceptionDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_len=256):
        self.texts = texts.values
        self.labels = labels.values if labels is not None else None
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])

        # 直接呼叫 tokenizer
        inputs = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        item = {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten()
        }

        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)

        return item

# 建立 DataLoader
train_dataset = MathMisconceptionDataset(train_data['input_text'], train_data['label'], tokenizer, MAX_LEN)
val_dataset = MathMisconceptionDataset(val_data['input_text'], val_data['label'], tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [4]:
# ================= Cell 4 =================
# 1. 檢查 DataLoader 狀態
print(f"訓練集 Batch 數量: {len(train_loader)}")
print(f"驗證集 Batch 數量: {len(val_loader)}")

if len(train_loader) == 0:
    print("⚠️ 警告：訓練集是空的！請回去重新執行 Cell 2 與 Cell 3。")
else:
    # 2. 初始化模型 (💡加入 ignore_mismatched_sizes=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, 
        num_labels=num_labels,
        ignore_mismatched_sizes=True
    )
    model = model.to(device)

    # 3. 優化器與 Learning Rate 排程設定
    EPOCHS = 4
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps*0.1), num_training_steps=total_steps)

    def train_epoch(model, dataloader, optimizer, scheduler, device):
        model.train()
        total_loss = 0
        correct_preds = 0

        for batch in tqdm(dataloader, desc="Training"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            model.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            correct_preds += torch.sum(preds == labels).item()

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

        return total_loss / len(dataloader), correct_preds / len(dataloader.dataset)

    def eval_model(model, dataloader, device):
        model.eval()
        total_loss = 0
        correct_preds = 0

        with torch.no_grad():
            for batch in tqdm(dataloader, desc="Evaluating"):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

                total_loss += outputs.loss.item()
                preds = torch.argmax(outputs.logits, dim=1)
                correct_preds += torch.sum(preds == labels).item()

        return total_loss / len(dataloader), correct_preds / len(dataloader.dataset)

    # 4. 開始訓練
    best_acc = 0
    for epoch in range(EPOCHS):
        print(f"\nEpoch {epoch+1}/{EPOCHS}")
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, device)
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")

        val_loss, val_acc = eval_model(model, val_loader, device)
        print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), 'best_mathbert.pt')
            print(">> Saved Best Model!")

訓練集 Batch 數量: 1950
驗證集 Batch 數量: 345


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: /kaggle/input/datasets/huangtzuchen/my-mathbert-weights/mathbert-local
Key               | Status   |                                                                                      
------------------+----------+--------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([2]) vs model:torch.Size([65])          
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([2, 768]) vs model:torch.Size([65, 768])

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.



Epoch 1/4


Training:   0%|          | 0/1950 [00:00<?, ?it/s]

...

Val Loss: 0.4639 | Val Acc: 0.8603
>> Saved Best Model!

Epoch 4/4


Training:   0%|          | 0/1950 [00:00<?, ?it/s]

Train Loss: 0.2337 | Train Acc: 0.9265


Evaluating:   0%|          | 0/345 [00:00<?, ?it/s]

Val Loss: 0.4487 | Val Acc: 0.8728
>> Saved Best Model!


In [5]:
# ================= Cell 5 =================
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

model.load_state_dict(torch.load('best_mathbert.pt'))
model.eval()

test_dataset = MathMisconceptionDataset(test_df['input_text'], tokenizer=tokenizer, max_len=MAX_LEN)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

all_top_3_preds = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Inference"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=-1)

        # 🎯 MAP@3 規定：最多預測 3 個
        top_3_indices = torch.topk(probs, 3, dim=1).indices.cpu().numpy()

        for indices in top_3_indices:
            # 🎯 直接反轉回我們在 Cell 2 建立的完整字串 (例如 "False_Misconception:WNB")
            top_3_labels = label_encoder.inverse_transform(indices)
            all_top_3_preds.append(" ".join(top_3_labels))

# 🎯 依照官方要求的欄位名稱建立 submission.csv
submission_df = pd.DataFrame({
    'row_id': test_df['row_id'] if 'row_id' in test_df.columns else test_df['QuestionId'].astype(str) + "_" + test_df['MC_Answer'].astype(str),
    'Category:Misconception': all_top_3_preds
})

submission_df.to_csv('submission.csv', index=False)
print("Submission saved to submission.csv")

Inference:   0%|          | 0/1 [00:00<?, ?it/s]

Submission saved to submission.csv
